# Advanced Twitter Sentiment Analysis & Emotion Intelligence
### Single-Notebook End-to-End NLP Project

**Dataset:** Twitter sentiment dataset (`train.csv` and `test.csv`)  
**Classes:** Positive, Neutral, Negative

This notebook covers data understanding, cleaning, NLP preprocessing, exploratory analysis, TF-IDF, multiple machine-learning models, feature engineering, model comparison, explainability, and real-time prediction.

> **Note:** The LSTM section is optional and is disabled by default because it can take considerably longer and requires TensorFlow. The main project is fully based on classical NLP/ML and can run without TensorFlow.


## 1. Project Objectives
- Build a complete sentiment-analysis pipeline.
- Clean and preprocess real-world tweets.
- Explore sentiment patterns visually.
- Compare several machine-learning algorithms.
- Add tweet-level linguistic features.
- Evaluate models using accuracy, precision, recall and F1-score.
- Explain individual predictions.
- Build an interactive prediction function.

**Fast Colab version:** uses 20,000 training tweets, 5,000 test tweets, and 10,000 TF-IDF features for substantially faster execution while keeping the complete project workflow.


In [ ]:
# 2. Import Libraries
import warnings
warnings.filterwarnings("ignore")

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print("Libraries loaded successfully.")


## 3. Load the Dataset
We use `train.csv` as the main training dataset and `test.csv` as the held-out evaluation dataset. This avoids accidentally mixing the supplied test labels into training.

In [ ]:
# ================================
# 3. LOAD DATASET - FAST COLAB VERSION
# ================================
import os
import zipfile
import shutil
import pandas as pd

DATA_DIR = "/content/twitter_dataset"
os.makedirs(DATA_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if not (os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH)):
    # Look for the CSVs (or a zip containing them) in a few common local
    # locations first, so this cell also works outside Google Colab.
    candidate_dirs = [".", "/content", "/mnt/user-data/uploads", DATA_DIR]
    found_locally = False
    for d in candidate_dirs:
        t_path = os.path.join(d, "train.csv")
        te_path = os.path.join(d, "test.csv")
        if os.path.exists(t_path) and os.path.exists(te_path):
            shutil.copyfile(t_path, TRAIN_PATH)
            shutil.copyfile(te_path, TEST_PATH)
            found_locally = True
            break
        zip_path = os.path.join(d, "archive.zip")
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, "r") as z:
                names = z.namelist()
                matches_ok = True
                for target in ["train.csv", "test.csv"]:
                    matches = [n for n in names if n == target or n.endswith("/" + target)]
                    if not matches:
                        matches_ok = False
                        break
                if matches_ok:
                    for target in ["train.csv", "test.csv"]:
                        matches = [n for n in names if n == target or n.endswith("/" + target)]
                        with z.open(matches[0]) as src, open(os.path.join(DATA_DIR, target), "wb") as dst:
                            shutil.copyfileobj(src, dst)
                    found_locally = True
                    break

    if not found_locally and _in_colab():
        from google.colab import files
        print("Upload your archive.zip containing train.csv and test.csv")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))

        with zipfile.ZipFile(zip_name, "r") as z:
            names = z.namelist()
            for target in ["train.csv", "test.csv"]:
                matches = [n for n in names if n == target or n.endswith("/" + target)]
                if not matches:
                    raise FileNotFoundError(f"{target} was not found inside the ZIP file.")
                with z.open(matches[0]) as src, open(os.path.join(DATA_DIR, target), "wb") as dst:
                    shutil.copyfileobj(src, dst)
        found_locally = True

    if not found_locally:
        raise FileNotFoundError(
            "Could not find train.csv/test.csv (or archive.zip containing them). "
            "Place them in the current directory, in /mnt/user-data/uploads, "
            "or (in Google Colab) re-run this cell to upload archive.zip."
        )

    print("Required dataset files ready!")

train_df = pd.read_csv(TRAIN_PATH, encoding="latin-1")
test_df = pd.read_csv(TEST_PATH, encoding="latin-1")

print("Original training shape:", train_df.shape)
print("Original testing shape :", test_df.shape)
display(train_df.head())


In [ ]:
# ================================
# FAST MODE CONFIGURATION
# ================================
# Downsample train_df/test_df in place so every later cell (cleaning, EDA,
# TF-IDF, modeling, feature engineering) works on the same reduced data.
# This is what actually delivers the "Fast Colab" speed-up described above.

MAX_TRAIN_SAMPLES = 20000
MAX_TEST_SAMPLES = 5000

full_train_shape = train_df.shape
full_test_shape = test_df.shape

if len(train_df) > MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(MAX_TRAIN_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

if len(test_df) > MAX_TEST_SAMPLES:
    test_df = test_df.sample(MAX_TEST_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

print("Training rows used:", len(train_df), "of", full_train_shape[0])
print("Testing rows used :", len(test_df), "of", full_test_shape[0])


## 4. Dataset Understanding

In [ ]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nMissing values:")
display(train_df.isnull().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", train_df.duplicated().sum())

print("\nSentiment distribution:")
display(train_df["sentiment"].value_counts())


In [ ]:
# Basic descriptive statistics
display(train_df.describe(include="all").T)

print("Unique sentiments:", train_df["sentiment"].unique())
print("Unique countries:", train_df["Country"].nunique())
print("Unique tweet IDs:", train_df["textID"].nunique())


## 5. Data Cleaning

In [ ]:
# Keep rows with a valid text and sentiment.
train_df = train_df.dropna(subset=["text", "sentiment"]).copy()
test_df = test_df.dropna(subset=["text", "sentiment"]).copy()

# Remove exact duplicate tweets from training data.
train_df = train_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Normalize labels to lowercase.
train_df["sentiment"] = train_df["sentiment"].str.lower().str.strip()
test_df["sentiment"] = test_df["sentiment"].str.lower().str.strip()

print("Clean training shape:", train_df.shape)
print("Clean testing shape :", test_df.shape)


## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Sentiment distribution
counts = train_df["sentiment"].value_counts()

plt.figure(figsize=(7, 5))
counts.plot(kind="bar")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Tweets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Tweet length analysis
train_df["raw_char_length"] = train_df["text"].astype(str).str.len()
train_df["raw_word_length"] = train_df["text"].astype(str).str.split().str.len()

display(train_df.groupby("sentiment")[["raw_char_length", "raw_word_length"]].mean().round(2))


In [ ]:
plt.figure(figsize=(9, 5))
for label in sorted(train_df["sentiment"].unique()):
    values = train_df.loc[train_df["sentiment"] == label, "raw_word_length"]
    plt.hist(values, bins=30, alpha=0.45, label=label)

plt.title("Tweet Word-Length Distribution by Sentiment")
plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Sentiment by time of tweet
time_sentiment = pd.crosstab(train_df["Time of Tweet"], train_df["sentiment"], normalize="index") * 100
display(time_sentiment.round(2))

time_sentiment.plot(kind="bar", figsize=(9, 5))
plt.title("Sentiment Percentage by Time of Tweet")
plt.xlabel("Time of Tweet")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Sentiment by age group
age_sentiment = pd.crosstab(train_df["Age of User"], train_df["sentiment"], normalize="index") * 100
display(age_sentiment.round(2))

age_sentiment.plot(kind="bar", figsize=(10, 5))
plt.title("Sentiment Percentage by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Percentage")
plt.tight_layout()
plt.show()


## 7. NLP Text Preprocessing
The function removes common noise from social-media text while preserving useful words. Negation words such as `not`, `no`, and `never` are deliberately kept.

In [ ]:
# Keep useful negation words while removing other English stopwords.
NEGATIONS = {"no", "not", "never", "neither", "nor", "cannot", "can't", "won't", "don't", "didn't", "isn't", "wasn't"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - NEGATIONS

def clean_text(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Remove mentions and hashtags symbols but keep hashtag words
    text = re.sub(r"[@#]([A-Za-z0-9_]+)", r"\1", text)

    # Remove HTML-like content
    text = re.sub(r"<.*?>", " ", text)

    # Lowercase
    text = text.lower()

    # Expand a few common contractions
    contractions = {
        "can't": "cannot", "won't": "will not", "n't": " not",
        "'re": " are", "'m": " am", "'ll": " will",
        "'ve": " have", "'d": " would", "'s": " is"
    }
    for old, new in contractions.items():
        text = text.replace(old, new)

    # Keep alphabetic words
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove stopwords and very short tokens
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 1]

    return " ".join(tokens)

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

display(train_df[["text", "clean_text", "sentiment"]].head(10))


## 8. Tokenization and Linguistic Features

In [ ]:
def tokenize(text):
    return text.split()

train_df["tokens"] = train_df["clean_text"].apply(tokenize)
test_df["tokens"] = test_df["clean_text"].apply(tokenize)

train_df["clean_word_count"] = train_df["tokens"].str.len()
train_df["exclamation_count"] = train_df["text"].str.count("!")
train_df["question_count"] = train_df["text"].str.count(r"\?")
train_df["mention_count"] = train_df["text"].str.count("@")
train_df["hashtag_count"] = train_df["text"].str.count("#")
train_df["url_count"] = train_df["text"].str.count(r"http")

display(train_df[["clean_word_count", "exclamation_count", "question_count",
                  "mention_count", "hashtag_count", "url_count"]].describe())


## 9. Most Frequent Words

In [ ]:
all_words = []
for tokens in train_df["tokens"]:
    all_words.extend(tokens)

word_counts = Counter(all_words)
top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])
display(top_words)


In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top_words["word"][::-1], top_words["frequency"][::-1])
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()


## 10. Simple Word Clouds
If `wordcloud` is installed, this cell creates a word cloud. If not, the project continues normally.

In [ ]:
try:
    from wordcloud import WordCloud

    for label in ["positive", "neutral", "negative"]:
        text = " ".join(train_df.loc[train_df["sentiment"] == label, "clean_text"])
        wc = WordCloud(width=900, height=450, background_color="white").generate(text)

        plt.figure(figsize=(12, 5))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"{label.title()} Word Cloud")
        plt.show()
except ImportError:
    print("Install wordcloud with: pip install wordcloud")


## 11. Train/Test Data Preparation

In [ ]:
X_train = train_df["clean_text"]
y_train = train_df["sentiment"]

X_test = test_df["clean_text"]
y_test = test_df["sentiment"]

print("Training examples:", len(X_train))
print("Testing examples :", len(X_test))


## 12. TF-IDF Feature Extraction
TF-IDF converts text into numerical features. `ngram_range=(1,2)` allows the model to learn both single words and two-word phrases.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF training matrix:", X_train_tfidf.shape)
print("TF-IDF testing matrix :", X_test_tfidf.shape)


## 13. Model 1 — Multinomial Naive Bayes

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

print(classification_report(y_test, nb_pred))


## 14. Model 2 — Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

print(classification_report(y_test, lr_pred))


## 15. Model 3 — Linear SVM

In [ ]:
svm_model = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

print(classification_report(y_test, svm_pred))


## 16. Model 4 — Random Forest
Random Forest is included as a comparison model. Sparse high-dimensional TF-IDF is usually better suited to linear models, so this model may be slower or less accurate.

In [ ]:
# Limit the number of features for Random Forest to keep runtime reasonable.
rf_tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_rf = rf_tfidf.fit_transform(X_train)
X_test_rf = rf_tfidf.transform(X_test)

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf_model.fit(X_train_rf, y_train)
rf_pred = rf_model.predict(X_test_rf)

print(classification_report(y_test, rf_pred))


## 17. Model 5 — XGBoost (Optional)
This section is optional because XGBoost may not be installed. The project does not depend on it.

In [ ]:
try:
    from xgboost import XGBClassifier

    # Encode labels for XGBoost
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    inv_label_map = {v: k for k, v in label_map.items()}

    X_train_xgb = X_train_tfidf
    X_test_xgb = X_test_tfidf
    y_train_xgb = y_train.map(label_map)

    xgb_model = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softmax",
        num_class=3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    xgb_model.fit(X_train_xgb, y_train_xgb)
    xgb_pred_num = xgb_model.predict(X_test_xgb)
    xgb_pred = pd.Series(xgb_pred_num).map(inv_label_map).values

    print(classification_report(y_test, xgb_pred))
    xgb_available = True
except Exception as e:
    print("XGBoost skipped:", e)
    xgb_available = False


## 18. Model Comparison

In [ ]:
predictions = {
    "Naive Bayes": nb_pred,
    "Logistic Regression": lr_pred,
    "Linear SVM": svm_pred,
    "Random Forest": rf_pred
}
if "xgb_available" in globals() and xgb_available:
    predictions["XGBoost"] = xgb_pred

results = []
for name, pred in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, pred, average="weighted", zero_division=0),
        "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(results_df.round(4))


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["F1 Score"])
plt.title("Model Comparison — Weighted F1")
plt.ylabel("F1 Score")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 19. Confusion Matrix for the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pred = predictions[best_model_name]

print("Best model:", best_model_name)
print(classification_report(y_test, best_pred))

cm = confusion_matrix(y_test, best_pred, labels=["negative", "neutral", "positive"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Neutral", "Positive"])
disp.plot(values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 20. Feature Engineering Experiment
We add simple social-media features to a Logistic Regression model. The goal is to test whether tweet structure adds information beyond TF-IDF.

In [ ]:
feature_cols = [
    "clean_word_count", "exclamation_count", "question_count",
    "mention_count", "hashtag_count", "url_count"
]

# Build the same features for test data
test_df["clean_word_count"] = test_df["clean_text"].str.split().str.len()
test_df["exclamation_count"] = test_df["text"].str.count("!")
test_df["question_count"] = test_df["text"].str.count(r"\?")
test_df["mention_count"] = test_df["text"].str.count("@")
test_df["hashtag_count"] = test_df["text"].str.count("#")
test_df["url_count"] = test_df["text"].str.count(r"http")

train_extra = train_df[feature_cols].fillna(0).astype(float).values
test_extra = test_df[feature_cols].fillna(0).astype(float).values

# Standardize the small numeric feature block
scaler = StandardScaler()
train_extra = scaler.fit_transform(train_extra)
test_extra = scaler.transform(test_extra)

from scipy.sparse import hstack
X_train_combined = hstack([X_train_tfidf, train_extra])
X_test_combined = hstack([X_test_tfidf, test_extra])

lr_feature_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
lr_feature_model.fit(X_train_combined, y_train)
lr_feature_pred = lr_feature_model.predict(X_test_combined)

print(classification_report(y_test, lr_feature_pred))
print("Accuracy:", round(accuracy_score(y_test, lr_feature_pred), 4))
print("Weighted F1:", round(f1_score(y_test, lr_feature_pred, average="weighted"), 4))


## 21. Full Tweet vs Selected Text Experiment
The dataset contains `selected_text`, which is the part of a tweet associated with the sentiment annotation. We can compare a model trained on the full tweet with one trained only on selected text.

In [ ]:
selected_train = train_df["selected_text"].fillna(train_df["text"]).apply(clean_text)
# test.csv does not contain selected_text, so this experiment uses a train/validation split.
X_sel_train, X_sel_val, y_sel_train, y_sel_val = train_test_split(
    selected_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

sel_vectorizer = TfidfVectorizer(max_features=40000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_sel_train_vec = sel_vectorizer.fit_transform(X_sel_train)
X_sel_val_vec = sel_vectorizer.transform(X_sel_val)

sel_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
sel_model.fit(X_sel_train_vec, y_sel_train)
sel_pred = sel_model.predict(X_sel_val_vec)

print("Selected-text validation results:")
print(classification_report(y_sel_val, sel_pred))


## 22. Explainable AI — Important Words
For Logistic Regression, each class has a coefficient for every TF-IDF feature. Large positive coefficients indicate words strongly associated with that class.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
classes = lr_model.classes_

for class_name in classes:
    class_index = list(classes).index(class_name)
    coef = lr_model.coef_[class_index]
    top_indices = np.argsort(coef)[-15:][::-1]
    print(f"\nTop words for {class_name.upper()}:")
    print(", ".join(feature_names[top_indices]))


## 23. Real-Time Sentiment Prediction

In [ ]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    vector = tfidf.transform([cleaned])

    # Use Logistic Regression because it provides class probabilities.
    prediction = lr_model.predict(vector)[0]
    probabilities = lr_model.predict_proba(vector)[0]

    prob_df = pd.DataFrame({
        "Sentiment": lr_model.classes_,
        "Probability": probabilities
    }).sort_values("Probability", ascending=False)

    print("Original text :", text)
    print("Cleaned text  :", cleaned)
    print("Prediction    :", prediction.upper())
    print("Confidence    :", f"{probabilities[list(lr_model.classes_).index(prediction)]*100:.2f}%")

    display(prob_df.style.format({"Probability": "{:.2%}"}))
    return prediction, prob_df

# Try your own tweet here:
predict_sentiment("I absolutely love this product! It is amazing.")


## 24. Batch Prediction

In [ ]:
sample_tweets = [
    "I love this, it is fantastic!",
    "This is the worst experience ever.",
    "The product arrived today.",
    "I am very happy with the service.",
    "I don't like this at all."
]

batch_vectors = tfidf.transform([clean_text(t) for t in sample_tweets])
batch_predictions = lr_model.predict(batch_vectors)
batch_probabilities = lr_model.predict_proba(batch_vectors).max(axis=1)

batch_results = pd.DataFrame({
    "Tweet": sample_tweets,
    "Predicted Sentiment": batch_predictions,
    "Confidence": batch_probabilities
})
display(batch_results.style.format({"Confidence": "{:.2%}"}))


## 25. Optional LSTM Deep-Learning Model
Run this only if TensorFlow is installed and you have enough CPU/GPU time. It is not required for the main project.

In [ ]:
RUN_LSTM = False

if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras.preprocessing.text import Tokenizer
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
        from sklearn.preprocessing import LabelEncoder

        # Use a manageable subset for a practical notebook run.
        MAX_LSTM_SAMPLES = min(30000, len(train_df))
        lstm_df = model_train_df.sample(MAX_LSTM_SAMPLES, random_state=RANDOM_STATE)

        le = LabelEncoder()
        y_lstm = le.fit_transform(lstm_df["sentiment"])

        tok = Tokenizer(num_words=20000, oov_token="<OOV>")
        tok.fit_on_texts(lstm_df["clean_text"])
        sequences = tok.texts_to_sequences(lstm_df["clean_text"])
        padded = pad_sequences(sequences, maxlen=80, padding="post", truncating="post")

        X_lstm_train, X_lstm_val, y_lstm_train, y_lstm_val = train_test_split(
            padded, y_lstm, test_size=0.2, random_state=RANDOM_STATE, stratify=y_lstm
        )

        lstm_model = Sequential([
            Embedding(20000, 128, input_length=80),
            LSTM(64),
            Dropout(0.4),
            Dense(32, activation="relu"),
            Dropout(0.3),
            Dense(3, activation="softmax")
        ])

        lstm_model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        history = lstm_model.fit(
            X_lstm_train, y_lstm_train,
            validation_data=(X_lstm_val, y_lstm_val),
            epochs=5,
            batch_size=128,
            verbose=1
        )

        lstm_prob = lstm_model.predict(X_lstm_val, verbose=0)
        lstm_pred_num = np.argmax(lstm_prob, axis=1)

        print(classification_report(y_lstm_val, lstm_pred_num, target_names=le.classes_))
    except Exception as e:
        print("LSTM could not be run:", e)
else:
    print("LSTM is disabled. Set RUN_LSTM = False to run it.")


## 26. Final Project Summary

In [ ]:
print("=" * 60)
print("FINAL PROJECT SUMMARY")
print("=" * 60)

print("Training samples :", len(train_df))
print("Testing samples  :", len(test_df))
print("Best ML model    :", best_model_name)
print("Best ML accuracy :", f"{accuracy_score(y_test, best_pred):.4f}")
print("Best ML F1 score :", f"{f1_score(y_test, best_pred, average='weighted'):.4f}")
print("Feature-engineered LR accuracy:",
      f"{accuracy_score(y_test, lr_feature_pred):.4f}")

print("\nSentiment classes:", sorted(train_df["sentiment"].unique()))
print("\nThe complete pipeline includes:")
print("✓ Data cleaning")
print("✓ Exploratory Data Analysis")
print("✓ NLP preprocessing")
print("✓ Tokenization and stopword handling")
print("✓ TF-IDF")
print("✓ Multiple ML models")
print("✓ Model comparison")
print("✓ Confusion matrix and classification metrics")
print("✓ Feature engineering")
print("✓ Selected-text experiment")
print("✓ Explainable model features")
print("✓ Real-time prediction")
print("✓ Optional LSTM")


# 27. Conclusion

This project demonstrates a complete end-to-end **Twitter Sentiment Analysis** system. The workflow starts with raw tweets, removes social-media noise, performs NLP preprocessing, converts text into TF-IDF features, trains several classifiers, compares their performance, and uses the strongest practical model for prediction.

The project also goes beyond basic sentiment classification by investigating tweet metadata, linguistic features, selected text, and model interpretability.

### Suggested final presentation points
1. Problem statement
2. Dataset and attributes
3. Data preprocessing
4. EDA findings
5. TF-IDF methodology
6. Models used
7. Evaluation metrics
8. Best model
9. Feature-engineering experiment
10. Explainability
11. Real-time prediction
12. Limitations and future work
